NEXT TIME YOU RUN THIS COMBINE THE LIABILITY LINK EXTRATCION INTO THE ACTUAL SCRAPING PIPE-LINE

In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs

import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display
from selenium import webdriver

# All paths are relative to the notebooks/ directory.
RAW_DIR = Path("../raw")
LINK_DIR = RAW_DIR / "lok_sabha_constituencies"
HTML_DIR = RAW_DIR / "lok_sabha_html"
OUT_DIR = RAW_DIR / "lok_sabha_candidate_dfs"
PAGE_CSV_DIR = OUT_DIR / "page_csvs"

for folder in [HTML_DIR, OUT_DIR, PAGE_CSV_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET_STATES = {
    "TAMIL NADU",
    "UTTAR PRADESH",
    "BIHAR",
    "MAHARASHTRA",
}

YEARS = [2024, 2019, 2014]

STABILITY_SECONDS = 1.25
POLL_SECONDS = 0.25
PAGE_TIMEOUT_SECONDS = 20

MANIFEST_PATH = HTML_DIR / "html_manifest.csv"
CANDIDATE_OUTPUT_PATH = OUT_DIR / "candidates_all_years.csv"

CANDIDATE_COLUMNS = [
    "year", "state", "state_id", "const", "const_id",
    "sno", "candidate", "candidate_id", "candidate_url", "is_winner",
    "party", "criminal_cases", "education", "age",
    "total_assets", "asset_link", "liabilities",
    "constituency_url", "html_path",
]

MANIFEST_COLUMNS = [
    "year", "state_ut", "state_id", "const", "const_id",
    "link", "election_date", "html_path", "candidate_csv_path",
    "status", "candidate_link_count", "parsed_candidate_count",
    "count_match", "stable", "elapsed_seconds", "error",
]

print("Working directory:", Path.cwd())
print("HTML output:", HTML_DIR.resolve())
print("Candidate output:", OUT_DIR.resolve())

In [ ]:
def clean_text(value):
    if value is None:
        return None
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value or None


def safe_filename(value):
    value = re.sub(r"[^A-Za-z0-9_-]+", "_", str(value).strip())
    return value.strip("_") or "unknown"


def unique_candidate_ids(driver):
    hrefs = driver.execute_script("""
        return Array.from(document.querySelectorAll("a[href]"))
            .map(a => a.href)
            .filter(href => /candidate\\.php/i.test(href));
    """)

    ids = set()

    for href in hrefs:
        params = parse_qs(urlparse(href).query)
        candidate_id = params.get("candidate_id", [None])[0]

        # Use the candidate ID when present; otherwise use the URL.
        ids.add(str(candidate_id) if candidate_id else href)

    return ids


def wait_for_stable_candidates(driver):
    start = time.monotonic()
    last_ids = set()
    stable_since = None
    stable = False

    while time.monotonic() - start < PAGE_TIMEOUT_SECONDS:
        try:
            current_ids = unique_candidate_ids(driver)
        except Exception:
            current_ids = set()

        now = time.monotonic()

        # Do not start the stability timer until candidates appear.
        if current_ids:
            if current_ids != last_ids:
                last_ids = current_ids
                stable_since = now
            elif stable_since is not None:
                if now - stable_since >= STABILITY_SECONDS:
                    stable = True
                    break

        time.sleep(POLL_SECONDS)

    elapsed = time.monotonic() - start

    # Recheck once at the end to capture the latest visible candidate links.
    try:
        final_ids = unique_candidate_ids(driver)
    except Exception:
        final_ids = set()

    return {
        "candidate_ids": final_ids,
        "candidate_link_count": len(final_ids),
        "stable": stable,
        "elapsed_seconds": round(elapsed, 2),
    }

In [ ]:
def header_key(text):
    return re.sub(r"[^a-z]", "", (text or "").lower())


def find_column(headers, possible_names):
    for index, header in enumerate(headers):
        key = header_key(header)
        if any(name in key for name in possible_names):
            return index
    return None


def get_cell_value(cells, index):
    if index is None or index >= len(cells):
        return None
    return clean_text(cells[index].get_text(" ", strip=True))


def parse_candidate_rows(html, page):
    soup = BeautifulSoup(html, "html.parser")
    rows_out = []

    for table in soup.find_all("table"):
        table_rows = table.find_all("tr")
        header_index = None
        headers = []

        # Find the row that actually contains candidate and asset headers.
        for i, tr in enumerate(table_rows):
            cells = tr.find_all(["th", "td"], recursive=False)
            current_headers = [
                clean_text(cell.get_text(" ", strip=True)) or ""
                for cell in cells
            ]
            keys = [header_key(h) for h in current_headers]

            has_candidate = any("candidate" in key for key in keys)
            has_assets = any(
                "totalassets" in key or "assets" in key for key in keys
            )

            if has_candidate and has_assets:
                header_index = i
                headers = current_headers
                break

        if header_index is None:
            continue

        candidate_col = find_column(headers, ["candidate"])
        party_col = find_column(headers, ["party"])
        criminal_col = find_column(headers, ["criminalcases", "criminalcase"])
        education_col = find_column(headers, ["education"])
        age_col = find_column(headers, ["age"])
        assets_col = find_column(headers, ["totalassets", "assets"])
        liabilities_col = find_column(headers, ["liabilities"])
        sno_col = find_column(headers, ["sno", "serialno", "serialnumber"])

        if candidate_col is None or assets_col is None:
            continue

        for tr in table_rows[header_index + 1:]:
            cells = tr.find_all(["td", "th"], recursive=False)

            if len(cells) <= max(candidate_col, assets_col):
                continue

            candidate_cell = cells[candidate_col]
            anchor = candidate_cell.find(
                "a", href=re.compile(r"candidate\.php", re.I)
            )

            if anchor is None:
                continue

            candidate_name = clean_text(anchor.get_text(" ", strip=True))
            candidate_url = urljoin(page["link"], anchor.get("href", ""))
            candidate_id = parse_qs(
                urlparse(candidate_url).query
            ).get("candidate_id", [None])[0]

            candidate_text = candidate_cell.get_text(" ", strip=True)
            is_winner = bool(
                re.search(r"\bWinner\b", candidate_text, flags=re.I)
            )

            asset_cell = cells[assets_col]
            asset_link = None

            # Preserve the asset declaration image/link when available.
            asset_element = asset_cell.find(["img", "a"])
            if asset_element is not None:
                asset_href = (
                    asset_element.get("src")
                    or asset_element.get("data-src")
                    or asset_element.get("href")
                )
                if asset_href:
                    asset_link = urljoin(page["link"], asset_href)

            rows_out.append({
                "year": page["year"],
                "state": page["state_ut"],
                "state_id": page["state_id"],
                "const": page["const"],
                "const_id": page["const_id"],
                "candidate": candidate_name,
                "candidate_id": (
                    str(candidate_id) if candidate_id is not None else None
                ),
                "sno": get_cell_value(cells, sno_col),
                "candidate_url": candidate_url,
                "is_winner": is_winner,
                "party": get_cell_value(cells, party_col),
                "criminal_cases": get_cell_value(cells, criminal_col),
                "education": get_cell_value(cells, education_col),
                "age": get_cell_value(cells, age_col),
                "total_assets": get_cell_value(cells, assets_col),
                "asset_link": asset_link,
                "liabilities": get_cell_value(cells, liabilities_col),
                "constituency_url": page["link"],
                "html_path": page["html_path"],
            })

    return rows_out

In [ ]:
link_frames = []

for year in YEARS:
    link_path = LINK_DIR / f"constituency_links_{year}.csv"

    if not link_path.exists():
        raise FileNotFoundError(
            f"Missing {link_path}. Run Notebook 1 first."
        )

    year_df = pd.read_csv(link_path, dtype={"const_id": str, "state_id": str})
    link_frames.append(year_df)

all_links = pd.concat(link_frames, ignore_index=True)

selected = all_links[
    all_links["state_ut"].fillna("").str.strip().str.upper().isin(TARGET_STATES)
].copy()

selected["year"] = selected["year"].astype(int)
selected["const_id"] = selected["const_id"].astype(str)

if selected["link"].duplicated().any():
    duplicate_urls = selected.loc[
        selected["link"].duplicated(keep=False), "link"
    ].tolist()
    raise ValueError(f"Duplicate constituency URLs found: {duplicate_urls[:5]}")

# Create deterministic output paths for every selected constituency.
def output_paths(row):
    state_folder = safe_filename(row["state_ut"])
    const_file = safe_filename(row["const"])
    base_name = f"{const_file}_{safe_filename(row['const_id'])}"

    html_path = HTML_DIR / str(row["year"]) / state_folder / f"{base_name}.html"
    csv_path = PAGE_CSV_DIR / str(row["year"]) / state_folder / f"{base_name}.csv"

    return html_path, csv_path


if MANIFEST_PATH.exists():
    old_manifest = pd.read_csv(MANIFEST_PATH, dtype={"const_id": str})
else:
    old_manifest = pd.DataFrame(columns=MANIFEST_COLUMNS)

manifest_by_link = {}

if not old_manifest.empty and "link" in old_manifest.columns:
    for record in old_manifest.to_dict("records"):
        if pd.notna(record.get("link")):
            manifest_by_link[str(record["link"])] = record

print("All constituency links:", len(all_links))
print("Selected target-state pages:", len(selected))
print("\nPages by year and state:")
display(
    selected.groupby(["year", "state_ut"])
    .size()
    .rename("constituency_pages")
    .reset_index()
)

In [ ]:
options = webdriver.ChromeOptions()
options.page_load_strategy = "none"
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
driver.implicitly_wait(0)

def save_manifest():
    manifest_df = pd.DataFrame(
        list(manifest_by_link.values())
    )

    # Keep the expected columns even if some values are missing.
    for column in MANIFEST_COLUMNS:
        if column not in manifest_df.columns:
            manifest_df[column] = None

    manifest_df = manifest_df[MANIFEST_COLUMNS]
    manifest_df.to_csv(MANIFEST_PATH, index=False)


try:
    total_pages = len(selected)

    for position, (_, row) in enumerate(selected.iterrows(), start=1):
        url = str(row["link"])
        html_path, candidate_csv_path = output_paths(row)

        html_path.parent.mkdir(parents=True, exist_ok=True)
        candidate_csv_path.parent.mkdir(parents=True, exist_ok=True)

        previous = manifest_by_link.get(url, {})

        # Skip only pages that were fully processed and saved.
        if (
            previous.get("status") == "ok"
            and html_path.exists()
            and candidate_csv_path.exists()
        ):
            print(
                f"[{position}/{total_pages}] SKIP "
                f"{row['year']} | {row['state_ut']} | {row['const']}"
            )
            continue

        page_info = {
            "year": int(row["year"]),
            "state_ut": row["state_ut"],
            "state_id": row.get("state_id"),
            "const": row["const"],
            "const_id": str(row["const_id"]),
            "link": url,
            "election_date": row.get("election_date"),
            "html_path": str(html_path),
            "candidate_csv_path": str(candidate_csv_path),
            "status": "error",
            "candidate_link_count": 0,
            "parsed_candidate_count": 0,
            "count_match": False,
            "stable": False,
            "elapsed_seconds": None,
            "error": None,
        }

        started = time.monotonic()

        try:
            driver.get(url)

            wait_result = wait_for_stable_candidates(driver)
            page_info["candidate_link_count"] = (
                wait_result["candidate_link_count"]
            )
            page_info["stable"] = wait_result["stable"]
            page_info["elapsed_seconds"] = wait_result["elapsed_seconds"]

            # Capture and save the rendered DOM, even if the page timed out.
            html = driver.page_source
            html_path.write_text(html, encoding="utf-8")

            page_metadata = {
                "year": int(row["year"]),
                "state_ut": row["state_ut"],
                "state_id": row.get("state_id"),
                "const": row["const"],
                "const_id": str(row["const_id"]),
                "link": url,
                "html_path": str(html_path),
            }

            candidate_records = parse_candidate_rows(html, page_metadata)
            page_candidates = pd.DataFrame(
                candidate_records,
                columns=CANDIDATE_COLUMNS,
            )

            # Save per-page CSV even when parsing returns zero rows.
            page_candidates.to_csv(candidate_csv_path, index=False)

            parsed_count = len(page_candidates)
            link_count = wait_result["candidate_link_count"]

            page_info["parsed_candidate_count"] = parsed_count
            page_info["count_match"] = (
                link_count > 0 and parsed_count == link_count
            )

            if not wait_result["stable"]:
                page_info["status"] = "unstable_timeout"
                page_info["error"] = (
                    "Candidate-link count did not stabilize before timeout."
                )
            elif link_count == 0:
                page_info["status"] = "no_candidate_links"
                page_info["error"] = "No candidate.php links detected."
            elif parsed_count == 0:
                page_info["status"] = "parse_error"
                page_info["error"] = (
                    "Candidate links were detected, but no candidate rows "
                    "were parsed from the rendered HTML."
                )
            elif parsed_count != link_count:
                page_info["status"] = "count_mismatch"
                page_info["error"] = (
                    f"Detected {link_count} unique candidate links but "
                    f"parsed {parsed_count} candidate rows."
                )
            else:
                page_info["status"] = "ok"

        except Exception as exc:
            page_info["status"] = "error"
            page_info["error"] = f"{type(exc).__name__}: {exc}"
            page_info["elapsed_seconds"] = round(
                time.monotonic() - started, 2
            )

        manifest_by_link[url] = page_info
        save_manifest()

        print(
            f"[{position}/{total_pages}] "
            f"{row['year']} | {row['state_ut']} | {row['const']} "
            f"| links={page_info['candidate_link_count']} "
            f"| parsed={page_info['parsed_candidate_count']} "
            f"| status={page_info['status']} "
            f"| {page_info['elapsed_seconds']}s"
        )

finally:
    driver.quit()
    save_manifest()

print("\nScraping pass finished.")
print("HTML manifest:", MANIFEST_PATH.resolve())

In [ ]:
candidate_frames = []

for record in manifest_by_link.values():
    csv_value = record.get("candidate_csv_path")
    if not csv_value or pd.isna(csv_value):
        continue

    csv_path = Path(str(csv_value))

    if not csv_path.exists():
        continue

    # Keep parsed records even for count mismatches so they can be inspected.
    if int(record.get("parsed_candidate_count") or 0) > 0:
        page_df = pd.read_csv(
            csv_path,
            dtype={
                "candidate_id": str,
                "const_id": str,
                "state_id": str,
            },
        )
        if not page_df.empty:
            candidate_frames.append(page_df)

if candidate_frames:
    candidates = pd.concat(candidate_frames, ignore_index=True)
    candidates = candidates.reindex(columns=CANDIDATE_COLUMNS)

    # Remove only exact candidate duplicates within the same election/
    # constituency; keep rows with missing candidate IDs using candidate URL.
    candidates["_dedup_key"] = candidates["candidate_id"].fillna("")
    missing_id = candidates["_dedup_key"].eq("")
    candidates.loc[missing_id, "_dedup_key"] = (
        candidates.loc[missing_id, "candidate_url"].fillna("")
    )

    candidates = candidates.drop_duplicates(
        subset=["year", "const_id", "_dedup_key"],
        keep="first",
    ).drop(columns="_dedup_key").reset_index(drop=True)

else:
    candidates = pd.DataFrame(columns=CANDIDATE_COLUMNS)

candidates.to_csv(CANDIDATE_OUTPUT_PATH, index=False)

manifest_df = pd.DataFrame(list(manifest_by_link.values()))

if not manifest_df.empty:
    for column in MANIFEST_COLUMNS:
        if column not in manifest_df.columns:
            manifest_df[column] = None
    manifest_df = manifest_df[MANIFEST_COLUMNS]

    print("Manifest status counts:")
    display(
        manifest_df["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="pages")
    )

    print("\nPages requiring inspection:")
    needs_inspection = manifest_df[
        ~manifest_df["status"].eq("ok")
        | ~manifest_df["count_match"].fillna(False).astype(bool)
    ]
    display(
        needs_inspection[
            [
                "year", "state_ut", "const", "const_id",
                "candidate_link_count", "parsed_candidate_count",
                "status", "error",
            ]
        ].reset_index(drop=True)
    )

print("\nFinal candidate rows:", len(candidates))
print("Unique candidate IDs:", candidates["candidate_id"].nunique())
print("Duplicate candidate keys:",
      candidates.duplicated(["year", "const_id", "candidate_id"]).sum())

print("\nCandidate rows by year and state:")
display(
    candidates.groupby(["year", "state"], dropna=False)
    .size()
    .rename("candidate_rows")
    .reset_index()
)

print("\nSaved candidate DataFrame:", CANDIDATE_OUTPUT_PATH.resolve())
print("Saved HTML manifest:", MANIFEST_PATH.resolve())

In [ ]:
# Number of repeated rows under the current candidate-ID definition
print("Total rows:", len(candidates))
print("Unique candidate IDs:", candidates["candidate_id"].nunique())
print("Rows with repeated candidate IDs:",
      candidates["candidate_id"].duplicated(keep=False).sum())

# Check uniqueness within each election and constituency
key = ["year", "const_id", "candidate_id"]

print("Duplicate rows for the same candidate in the same constituency/year:",
      candidates.duplicated(subset=key, keep=False).sum())

# Inspect the most frequently repeated candidate IDs
display(
    candidates.groupby("candidate_id", dropna=False)
    .agg(
        rows=("candidate_id", "size"),
        years=("year", "nunique"),
        constituencies=("const_id", "nunique"),
        names=("candidate", "nunique")
    )
    .query("rows > 1")
    .sort_values("rows", ascending=False)
    .head(20)
)

In [ ]:
import pandas as pd
import re
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs

HTML_DIR = Path("../raw/lok_sabha_html")
OUT_DIR = Path("../raw/lok_sabha_candidate_dfs")
MANIFEST_PATH = HTML_DIR / "html_manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH.resolve()}")

manifest = pd.read_csv(MANIFEST_PATH, dtype={"const_id": "string"})

print("Saved HTML pages:", len(manifest))
print("Manifest statuses:")
display(manifest["status"].value_counts(dropna=False).to_frame("pages"))

In [ ]:
def header_key(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def find_column(headers, possible_names):
    for i, header in enumerate(headers):
        key = header_key(header)
        if any(name in key for name in possible_names):
            return i
    return None


def get_cell_value(cells, index):
    if index is None or index >= len(cells):
        return None

    value = cells[index].get_text(" ", strip=True)
    return value if value else None


def get_cell_link(cell, base_url):
    """
    Return the first useful URL found in a cell.

    Prefer the actual image URL, then an enclosing/ordinary anchor URL.
    This is independent of the text in the cell.
    """
    if cell is None:
        return None

    # Direct image URLs, including common lazy-loading attributes.
    for img in cell.find_all("img"):
        for attr in ["src", "data-src", "data-original", "data-lazy-src"]:
            value = img.get(attr)
            if value and value.strip():
                return urljoin(base_url, value.strip())

        # srcset may contain multiple candidates; use the first URL.
        srcset = img.get("srcset")
        if srcset:
            first_candidate = srcset.split(",")[0].strip().split()[0]
            if first_candidate:
                return urljoin(base_url, first_candidate)

    # If there is no image URL, preserve a linked resource if present.
    for anchor in cell.find_all("a", href=True):
        href = anchor["href"].strip()
        if href and not href.startswith(("javascript:", "#")):
            return urljoin(base_url, href)

    return None


def get_cell(cells, index):
    if index is None or index >= len(cells):
        return None
    return cells[index]

In [16]:
candidate_rows = []
page_parse_results = []

successful_pages = manifest.loc[manifest["status"].eq("ok")]

for _, page in successful_pages.iterrows():
    html_path = Path(str(page["html_path"]))

    if not html_path.exists():
        print("Missing saved HTML:", html_path)
        page_parse_results.append({
            "year": page["year"],
            "const_id": page["const_id"],
            "html_path": str(html_path),
            "status": "missing_html",
            "candidate_rows": 0,
            "liability_links_found": 0,
        })
        continue

    try:
        soup = BeautifulSoup(html_path.read_bytes(), "html.parser")
    except Exception as e:
        print("Could not read:", html_path, "|", e)
        page_parse_results.append({
            "year": page["year"],
            "const_id": page["const_id"],
            "html_path": str(html_path),
            "status": "read_error",
            "candidate_rows": 0,
            "liability_links_found": 0,
        })
        continue

    page_rows = []

    for table in soup.find_all("table"):
        header_row = table.find("tr")
        if header_row is None:
            continue

        headers = [
            cell.get_text(" ", strip=True)
            for cell in header_row.find_all(["th", "td"], recursive=False)
        ]

        candidate_col = find_column(headers, ["candidate"])
        sno_col = find_column(headers, ["sno", "serialno", "serialnumber"])
        party_col = find_column(headers, ["party"])
        criminal_col = find_column(headers, ["criminalcases", "criminalcase"])
        education_col = find_column(headers, ["education"])
        age_col = find_column(headers, ["age"])
        assets_col = find_column(headers, ["totalassets", "assets"])
        liabilities_col = find_column(headers, ["liabilities", "liability"])

        # Identify the candidate table.
        if candidate_col is None or assets_col is None:
            continue

        for tr in table.find_all("tr")[1:]:
            cells = tr.find_all(["td", "th"], recursive=False)

            if len(cells) <= max(candidate_col, assets_col):
                continue

            candidate_cell = cells[candidate_col]
            candidate_anchor = candidate_cell.find(
                "a", href=re.compile(r"candidate\.php", re.I)
            )

            if candidate_anchor is None:
                continue

            candidate_name = candidate_anchor.get_text(" ", strip=True)
            candidate_url = urljoin(page["link"], candidate_anchor["href"])
            candidate_id = parse_qs(
                urlparse(candidate_url).query
            ).get("candidate_id", [None])[0]

            candidate_text = candidate_cell.get_text(" ", strip=True)
            is_winner = bool(
                re.search(r"\bWinner\b", candidate_text, flags=re.I)
            )

            asset_cell = get_cell(cells, assets_col)
            liability_cell = get_cell(cells, liabilities_col)

            # Extract these independently. Neither requires the other to exist.
            asset_link = get_cell_link(asset_cell, page["link"])
            liability_link = get_cell_link(liability_cell, page["link"])

            row = {
                "year": page["year"],
                "state": page["state_ut"],
                "state_id": page.get("state_id"),
                "const": page["const"],
                "const_id": page["const_id"],
                "sno": get_cell_value(cells, sno_col),
                "candidate": candidate_name,
                "candidate_id": candidate_id,
                "candidate_url": candidate_url,
                "is_winner": is_winner,
                "party": get_cell_value(cells, party_col),
                "criminal_cases": get_cell_value(cells, criminal_col),
                "education": get_cell_value(cells, education_col),
                "age": get_cell_value(cells, age_col),
                "total_assets": get_cell_value(cells, assets_col),
                "asset_link": asset_link,
                "liabilities": get_cell_value(cells, liabilities_col),
                "liability_link": liability_link,
                "constituency_url": page["link"],
                "html_path": str(html_path),
            }

            page_rows.append(row)

        # Stop after parsing the first matching candidate table on this page.
        if page_rows:
            break

    candidate_rows.extend(page_rows)

    page_parse_results.append({
        "year": page["year"],
        "const_id": page["const_id"],
        "html_path": str(html_path),
        "status": "parsed" if page_rows else "no_candidate_rows",
        "candidate_rows": len(page_rows),
        "liability_links_found": sum(
            bool(row["liability_link"]) for row in page_rows
        ),
    })

    print(
        f'{page["year"]} | {page["const"]}: '
        f'{len(page_rows)} candidates, '
        f'{sum(bool(row["liability_link"]) for row in page_rows)} liability links'
    )

candidates = pd.DataFrame(candidate_rows)

print("\nTotal parsed candidate rows:", len(candidates))
print("Rows with liability text:", candidates["liabilities"].notna().sum())
print("Rows with liability links:", candidates["liability_link"].notna().sum())

2019 | SHIRDI: 20 candidates, 6 liability links
2019 | SHIRUR: 23 candidates, 7 liability links
2019 | SOLAPUR: 13 candidates, 4 liability links
2019 | THANE: 23 candidates, 7 liability links
2019 | WARDHA: 14 candidates, 4 liability links
2019 | YAVATMAL WASHIM: 23 candidates, 7 liability links
2019 | ARAKKONAM: 15 candidates, 5 liability links
2019 | ARANI: 14 candidates, 4 liability links
2019 | CHENNAI CENTRAL: 31 candidates, 10 liability links
2019 | CHENNAI NORTH: 23 candidates, 7 liability links
2019 | CHENNAI SOUTH: 39 candidates, 13 liability links
2019 | CHIDAMBARAM: 11 candidates, 3 liability links
2019 | COIMBATORE: 14 candidates, 4 liability links
2019 | CUDDALORE: 21 candidates, 7 liability links
2019 | DHARMAPURI: 15 candidates, 5 liability links
2019 | DINDIGUL: 18 candidates, 6 liability links
2019 | ERODE: 20 candidates, 6 liability links
2019 | KALLAKURICHI: 21 candidates, 7 liability links
2019 | KANCHEEPURAM: 10 candidates, 3 liability links
2019 | KANNIYAKUMARI: 1

In [18]:
liability_text_present = (
    candidates["liabilities"].notna()
    & candidates["liabilities"].astype("string").str.strip().ne("")
)

liability_link_present = (
    candidates["liability_link"].notna()
    & candidates["liability_link"].astype("string").str.strip().ne("")
)

# The actual failure condition you want to investigate.
both_missing = ~liability_text_present & ~liability_link_present

print("Total candidate rows:", len(candidates))
print("Both liability text and link missing:", both_missing.sum())
print("Text missing, link present:", (~liability_text_present & liability_link_present).sum())
print("Text present, link missing:", (liability_text_present & ~liability_link_present).sum())
print("Both present:", (liability_text_present & liability_link_present).sum())

display(
    candidates.loc[
        both_missing,
        [
            "year", "state", "const", "const_id", "sno",
            "candidate", "candidate_id", "liabilities",
            "liability_link", "constituency_url", "html_path",
        ],
    ]
)

Total candidate rows: 10395
Both liability text and link missing: 0
Text missing, link present: 3260
Text present, link missing: 7135
Both present: 0


,year,state,const,const_id,sno,candidate,candidate_id,liabilities,liability_link,constituency_url,html_path


Adding date to the candidates data files

In [ ]:
from pathlib import Path
import pandas as pd
import re

RAW = Path("../raw")
CAND_DIR = RAW / "lok_sabha_candidate_dfs"
LINK_PATH = RAW / "lok_sabha_constituencies" / "constituency_links_all_years.csv"

YEARS = {"2014", "2019", "2024"}
STATES = {"BIHAR", "MAHARASHTRA", "TAMIL_NADU", "UTTAR_PRADESH"}

links = pd.read_csv(LINK_PATH, dtype=str).fillna("")

def norm(x):
    return re.sub(r"[^a-z0-9]", "", str(x).casefold())

def url_key(s):
    return s.astype(str).str.strip().str.rstrip("/").str.casefold()

links["_url"] = url_key(links["link"])
links["_year"] = links["year"].map(norm)
links["_state"] = links["state_ut"].map(norm)
links["_const"] = links["const"].map(norm)
links["_const_id"] = links["const_id"].map(norm)

links["is_byelection"] = links["is_byelection"].map(
    lambda x: str(x).strip().casefold() in {"true", "1", "yes", "y"}
)

# A URL must not map to conflicting regular-election dates.
regular_links = links[~links["is_byelection"]].copy()
conflicts = regular_links.groupby("_url")["election_date"].nunique()
if (conflicts > 1).any():
    raise ValueError("Conflicting regular-election dates found for the same URL.")

by_url = (
    regular_links.drop_duplicates("_url")
    .set_index("_url")["election_date"]
    .to_dict()
)

# Fallback for files without a matching constituency URL.
by_identity = {}
for _, r in regular_links.iterrows():
    key = (r["_year"], r["_state"], r["_const"], r["_const_id"])
    date = r["election_date"]
    if key in by_identity and by_identity[key] != date:
        raise ValueError(f"Conflicting dates for constituency key: {key}")
    by_identity[key] = date

files = [CAND_DIR / "candidates_all_years.csv"]

# Per-year/state candidate CSVs.
for year in YEARS:
    for state in STATES:
        p = CAND_DIR / year / state / "candidates.csv"
        if p.exists():
            files.append(p)

# Individual constituency-page CSVs.
page_dir = CAND_DIR / "page_csvs"
if page_dir.exists():
    files.extend(
        p for p in page_dir.rglob("*.csv")
        if len(p.relative_to(page_dir).parts) >= 3
        and p.relative_to(page_dir).parts[0] in YEARS
        and p.relative_to(page_dir).parts[1] in STATES
    )

updated = 0

for path in dict.fromkeys(files):
    if not path.exists():
        continue

    df = pd.read_csv(path, dtype=str, keep_default_na=False)

    if "constituency_url" in df.columns:
        keys = url_key(df["constituency_url"])
        dates = keys.map(by_url)
    else:
        dates = pd.Series("", index=df.index, dtype=str)

    # Identity fallback for rows whose URL did not match.
    required = {"year", "state", "const", "const_id"}
    if required.issubset(df.columns):
        for i in df.index[dates.isna() | dates.eq("")]:
            key = (
                norm(df.at[i, "year"]),
                norm(df.at[i, "state"]),
                norm(df.at[i, "const"]),
                norm(df.at[i, "const_id"]),
            )
            dates.at[i] = by_identity.get(key, "")

    if "is_byelection" in df.columns:
        is_byelection = df["is_byelection"].map(
            lambda x: str(x).strip().casefold() in {"true", "1", "yes", "y"}
        )
    else:
        # Infer by-election status from the matched constituency URL.
        bye_by_url = links.drop_duplicates("_url").set_index("_url")["is_byelection"].to_dict()
        is_byelection = url_key(
            df["constituency_url"] if "constituency_url" in df.columns
            else pd.Series("", index=df.index)
        ).map(bye_by_url).fillna(False)

    df["election_date"] = dates.fillna("").where(~is_byelection, "")
    df.to_csv(path, index=False)
    updated += 1
    print(f"Updated: {path}")

print(f"\nFinished. Updated {updated} CSV files.")

We initially implemented constituency and candidate scraping using Requests and BeautifulSoup, but testing revealed that although Requests reliably retrieved the constituency links, it often returned incomplete candidate tables—frequently only one candidate when Selenium's live DOM contained many (e.g., 42 candidates for Varanasi in 2014). We therefore retained Requests + BeautifulSoup for `scrape_const`, where it matched Selenium's constituency URL sets across all three years (1,679 links total), and switched to Selenium for candidate extraction. To avoid fixed, unnecessarily long waits, we tested problematic constituency pages and observed how their candidate-link counts changed over time; a 1.25-second stability window, polling every 0.25 seconds, was selected because the tested pages produced stable counts without changes during extended observation, with a 20-second maximum timeout as a safeguard. We used headless Chrome with `page_load_strategy="none"` so navigation would not block on the browser's normal page-load condition, then explicitly waited for the candidate IDs to stabilize. Afterward, we saved each rendered HTML snapshot for reproducibility and offline reparsing, used BeautifulSoup to identify the candidate table by its headers and extract S.No., candidate identity and profile URL, winner status, party, criminal cases, education, age, assets and liabilities, and stored constituency metadata alongside each record. We save per-constituency CSVs and a manifest containing candidate-link counts, parsed-row counts, stability, status, timings and errors; compare link counts with parsed-row counts to flag potential extraction failures; and combine the page-level CSVs into a final dataset. Finally, the manifest and existing files support resumability, so successful pages can be skipped on subsequent runs. Candidate extraction is restricted to Tamil Nadu, Uttar Pradesh, Bihar and Maharashtra, while the constituency-link dataset retains all states and Union Territories.